In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import wandb
import torch
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from tqdm import tqdm
import os
from fractal_sweep_config import sweep_config

In [2]:
# importing local modules
from src.activations import (relu, squared_relu, cubic_relu,
                              d_relu, d_squared_relu, d_cubic_relu)
from src.base_functions import (relu_H5, relu_H5_d, relu_H5_dd,
                                squared_relu_H5, squared_relu_H5_d, squared_relu_H5_dd,
                                cubic_relu_H5, cubic_relu_H5_d, cubic_relu_H5_dd)
from src.fractal_functions import (pointwise_fractal, alpha_fractalize, alpha_fractalize_first_derivative)
from src.fractal_activation import FractalActivation
from src.tools import get_transforms

In [3]:
a = -1
b = 1
n_subintervals = 6
n_iter = 2
alpha = [0, 0, 0, 0.1, 0.2, 0.3]

In [10]:
# Precompute fractal activation functions and their derivatives.
# The H5 base functions are evaluated with x1=a, xN=b to match the fractal domain.

relu_fractal = alpha_fractalize(relu, lambda z: relu_H5(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)
d_relu_fractal = alpha_fractalize_first_derivative(d_relu, lambda z: relu_H5_d(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)

squared_relu_fractal = alpha_fractalize(squared_relu, lambda z: squared_relu_H5(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)
d_squared_relu_fractal = alpha_fractalize_first_derivative(d_squared_relu, lambda z: squared_relu_H5_d(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)

cubic_relu_fractal = alpha_fractalize(cubic_relu, lambda z: cubic_relu_H5(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)
d_cubic_relu_fractal = alpha_fractalize_first_derivative(d_cubic_relu, lambda z: cubic_relu_H5_d(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)

f(-1) = 0.0, f(1) = 1.0
g(-1) = 0.0, g(1) = 1.0
f(-1) = 0.0, f(1) = 1.0
g(-1) = 0.0, g(1) = 1.0
f(-1) = 0.0, f(1) = 1.0
g(-1) = 0.0, g(1) = 1.0


In [11]:
# Helper function to get a fractal activation nn.Module by name.
# Returns an nn.Module that can be used in nn.Sequential.
# Inside [-1, 1]: uses fractal interpolation. Outside: uses classical activation.
def get_activation(name):
    name = name.lower()
    if name == 'f_relu':
        return FractalActivation(relu_fractal, lambda x: F.relu(x))
    if name == 'f_squared_relu':
        return FractalActivation(squared_relu_fractal, lambda x: F.relu(x) ** 2)
    if name == 'f_cubic_relu':
        return FractalActivation(cubic_relu_fractal, lambda x: F.relu(x) ** 3)
    raise ValueError(f"Unsupported activation: {name}")


class CNNModel(nn.Module):      
    def __init__(self, 
                 filters,                    # List of filters => Controls number of conv layers and filter sizes  
                 kernel_size,                # Size of filters  
                 activation,                 # Activation function  
                 dropout,                    # Dropout rate (optional)
                 use_batchnorm,              # Whether to use batch norm (optional)
                 input_shape=(3, 256, 256),  # Input shape compatible with iNaturalist dataset  
                 dense_units=256,            # Number of neurons in the dense (fully connected) layer  
                 num_classes=10):            # Output layer with 10 neurons  

        super().__init__()
        layers = []
        in_channels = input_shape[0]

        # Building conv-activation-maxpool blocks 
        for out_channels in filters:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1))  # Conv layer
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))   # Optional BatchNorm
            layers.append(get_activation(activation))         # Fractal activation module 
            layers.append(nn.MaxPool2d(2))                    # Max pooling 
            if dropout > 0:
                layers.append(nn.Dropout(dropout))            # Optional dropout
            in_channels = out_channels

        # Feature extractor with conv-activation-maxpool blocks 
        self.features = nn.Sequential(*layers)

        # Automatically calculate the output size after conv layers for the FC layer
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            out = self.features(dummy)
            flatten_size = out.view(1, -1).shape[1]

        # Classifier block with dense + activation + dropout + output 
        self.classifier = nn.Sequential(
            nn.Linear(flatten_size, dense_units),   # First dense layer  
            get_activation(activation),             # Fractal activation in dense layer  
            nn.Dropout(dropout),                    # Dropout
            nn.Linear(dense_units, num_classes)     # Output layer with 10 neurons  
        )

    def forward(self, x):
        x = self.features(x)            # Pass through convolutional blocks
        x = torch.flatten(x, 1)         # Flatten before fully connected layers
        return self.classifier(x)       # Output logits for classification

In [12]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/amar-kumar/.netrc
wandb: Currently logged in as: amar743840 (Fractal_Networks) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
# Training function
def train():
    # Initialize wandb
    wandb.init()
    config = wandb.config
    # Generating a meaningful run name using config values
    # run_name = f"run_filters-{config.filters_per_layer}_act-{config.activation}_bs-{config.batch_size}_lr-{config.learning_rate}_do-{config.dropout_rate}_bn-{config.use_batchnorm}_aug-{config.augmentation}"
    # wandb.run.name = run_name
    # wandb.run.save()

    # Transforms
    train_tf, val_tf = get_transforms(config.augmentation)

    # Loading datasets
    train_data = datasets.ImageFolder(str(PROJECT_ROOT / "inaturalist_12K" / "train"), transform=train_tf)
    val_data = datasets.ImageFolder(str(PROJECT_ROOT / "inaturalist_12K" / "val"), transform=val_tf)

    train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size=config.batch_size, shuffle=False, num_workers=2)

    # Preparing model
    filters = config.filters_per_layer
    model = CNNModel(
        filters=filters,
        kernel_size=3,
        activation=config.activation,
        dropout=config.dropout_rate,
        use_batchnorm=config.use_batchnorm,
        input_shape=(3, 256, 256)  
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Loss & optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # To track best validation accuracy
    best_val_acc = 0.0

    # Training loop
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        print("-" * 60)
        model.train()
        total_loss, correct, total = 0, 0, 0

        for inputs, labels in tqdm(train_loader, desc="Training Progress", ncols=100, colour="magenta"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        # Validation loop
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation Progress", ncols=100, colour="cyan"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
        print("-" * 60)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

    # Saving model if it's the best across all sweeps
    global_best_path = "best_accuracy.txt"
    current_best = 0.0
    
    # Reading global best accuracy if file exists
    if os.path.exists(global_best_path):
        with open(global_best_path, "r") as f:
            try:
                current_best = float(f.read().strip())
            except:
                current_best = 0.0
    
    # Saving model only if it's better than global best
    if val_acc > current_best:
        torch.save(model.state_dict(), "best_model.pth")
        with open(global_best_path, "w") as f:
            f.write(str(val_acc))
        print(f"New global best model saved with val_acc: {val_acc:.4f}")

    wandb.finish()  
    print("Training run complete.")

In [ ]:
if __name__ == "__main__":
    sweep_id = wandb.sweep(sweep_config, project="fractal_CNN")
    wandb.agent(sweep_id, function=train, count = 2)
    print("Sweep complete")

Create sweep with ID: 4etxl1gf
Sweep URL: https://wandb.ai/Fractal_Networks/Fractal_CNN/sweeps/4etxl1gf


wandb: Agent Starting Run: kzu6dsgu with config:
wandb: 	activation: f_relu
wandb: 	augmentation: True
wandb: 	batch_size: 32
wandb: 	dense_units: 128
wandb: 	dropout_rate: 0.2
wandb: 	epochs: 10
wandb: 	filters_per_layer: [32, 32, 32, 32, 32]
wandb: 	learning_rate: 0.001
wandb: 	use_batchnorm: True
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/amar-kumar/.netrc.



Epoch 1/10
------------------------------------------------------------


Validation Progress: 100%|██████████████████████████████████████████| 63/63 [02:17<00:00,  2.18s/it]


Train Loss: 2.2140, Train Acc: 18.89%
Val Loss: 2.1333, Val Acc: 22.65%
------------------------------------------------------------

Epoch 2/10
------------------------------------------------------------


...

Train Loss: 1.9445, Train Acc: 31.67%
Val Loss: 2.0487, Val Acc: 25.10%
------------------------------------------------------------

Epoch 8/10
------------------------------------------------------------


Training Progress:  65%|███████████████████████████▍              | 163/250 [08:52<05:14,  3.62s/it]wandb: Ctrl + C detected. Stopping sweep.


Sweep complete
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f463f2f9460>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f463e2b2330, execution_count=14 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f463e2b2210, raw_cell="if __name__ == "__main__":
    sweep_id = wandb.sw.." transformed_cell="if __name__ == "__main__":
    sweep_id = wandb.sw.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/amar-kumar/Desktop/Fractal/NN/Fractal_CNN/fractal/fractal_train.ipynb#X10sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

Training Progress:  66%|███████████████████████████▌              | 164/250 [08:58<04:42,  3.28s/it]Traceback (most recent call last):
  File "/home/amar-kumar/Desktop/Fractal/frctl_env/lib/python3.12/site-packages/tqdm/std.py", line 1191, in __iter__
    self.update(n - last_print_n)
  File "/home/amar-kumar/Desktop/Fractal/frctl_env/lib/python3.12/site-packages/tqdm/std.py", line 1242, in update
    self.refresh(lock_args=self.lock_args)
  File "/home/amar-kumar/Desktop/Fractal/frctl_env/lib/python3.12/site-packages/tqdm/std.py", line 1347, in refresh
    self.display()
  File "/home/amar-kumar/Desktop/Fractal/frctl_env/lib/python3.12/site-packages/tqdm/std.py", line 1495, in display
    self.sp(self.__str__() if msg is None else msg)
  File "/home/amar-kumar/Desktop/Fractal/frctl_env/lib/python3.12/site-packages/tqdm/std.py", line 459, in print_status
    fp_write('\r' + s + (' ' * max(last_len[0] - len_s, 0)))
  File "/home/amar-kumar/Desktop/Fractal/frctl_env/lib/python3.12/site-p